In [ ]:
from TerraFin import configure, load_terrafin_config
from TerraFin.data.providers.corporate.filings.sec_edgar import (
    build_toc,
    download_filing,
    get_company_filings,
    get_sec_toc,
    parse_sec_filing,
    ticker_to_cik_dict,
)


configure()
config = load_terrafin_config()

"""
1. Get company CIK mappings
2. Retrieve SEC filings for a company
3. Download and parse SEC filing content
"""

cik = ticker_to_cik_dict().get("AAPL")

filings_df = get_company_filings(cik)

print(filings_df.head())

In [ ]:
# Get filing details
filing_index = 0  # latest filing
accession_number = filings_df.accessionNumber.iloc[filing_index].replace("-", "")
file_name = filings_df.primaryDocument.iloc[filing_index]
filing_form = filings_df.primaryDocDescription.iloc[filing_index]

# Download content
html_content = download_filing(cik, accession_number, file_name)

In [ ]:
# Non-parsed html
from IPython.display import HTML


HTML(html_content)

In [ ]:
# Parsed markdown (default: images excluded to keep the LLM token budget small).
from IPython.display import Markdown


parsed_content = parse_sec_filing(html_content, filing_form)
Markdown(parsed_content)

In [ ]:
# Parsed markdown with images included.
# - `<img src="..." alt="...">` tags become `![alt](src)`.
# - base64 data URIs are replaced with `<inline-image:{mime}>` placeholders
#   so a 100 KB inline PNG doesn't land in the LLM context.
# - `alt` text is whitespace-collapsed and truncated to 120 chars to
#   mitigate prompt-injection via attacker-crafted alt text.
parsed_with_images = parse_sec_filing(html_content, filing_form, include_images=True)
Markdown(parsed_with_images)

In [ ]:
# Table of contents extracted from already-parsed markdown.
# Default is compact (top-level ## only) to save agent context; each entry
# carries a char_count so the agent can budget before asking for a section body.
toc = build_toc(parsed_content)
for entry in toc:
    print(f"{entry['text']:<60s}  ({entry['char_count']:>7,} chars, line {entry['line_index']})")

# Pass max_level=None (or a larger number) to drill into sub-sections.
# full_toc = build_toc(parsed_content, max_level=None)

In [ ]:
# One-shot path — reuses the parsed-markdown file cache under `sec_filings`,
# so repeat calls for the same filing skip both the SEC download and the
# sec_parser semantic walk.
toc = get_sec_toc("AAPL", filing_type="10-Q")

# Compact by default (max_level=2). The agent sees the canonical Part/Item
# scaffold with body sizes, then decides which section(s) to actually read.
for entry in toc:
    print(f"- {entry['text']}  ({entry['char_count']:,} chars) [slug={entry['slug']}]")